# Microsoft Fabric: SQL Analytics Endpoint Metadata Sync

## Problem
When you write Delta tables to a Fabric **Lakehouse** via Spark (notebooks, pipelines, dataflows), the data does not immediately appear in the **SQL analytics endpoint**. Fabric runs an automatic metadata sync process, but it can be delayed — sometimes by minutes — especially when:

- A large number of small Parquet files exist
- Complex partitioning schemes are used
- Heavy concurrent workloads cause internal sync queuing
- Tables use data types unsupported by the SQL analytics endpoint (e.g., `ARRAY`, `MAP`, `STRUCT`)

## Solution
This notebook uses the **official Fabric REST API** to:

1. **Discover** the SQL endpoint ID from a Lakehouse
2. **Force** a metadata refresh via `POST /v1/workspaces/{id}/sqlEndpoints/{id}/refreshMetadata`
3. **Poll** the Long Running Operation (LRO) until completion
4. **Report** per-table sync status, including errors and warnings

### API Reference
| Endpoint | Docs |
|---|---|
| Refresh SQL Endpoint Metadata | [learn.microsoft.com](https://learn.microsoft.com/en-us/rest/api/fabric/sqlendpoint/items/refresh-sql-endpoint-metadata) |
| Get Lakehouse | [learn.microsoft.com](https://learn.microsoft.com/en-us/rest/api/fabric/lakehouse/items/get-lakehouse) |
| LRO Pattern | [learn.microsoft.com](https://learn.microsoft.com/en-us/rest/api/fabric/articles/long-running-operation) |
| FabricRestClient | [learn.microsoft.com](https://learn.microsoft.com/en-us/python/api/semantic-link-sempy/sempy.fabric.fabricrestclient) |

---
## 1. Imports & Configuration

In [ ]:
import json
import time
import logging
from datetime import datetime, timezone

import sempy.fabric as fabric
import pandas as pd

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
TIMEOUT_UNIT = "Minutes"     # Seconds | Minutes | Hours | Days
TIMEOUT_VALUE = 15           # How long Fabric will attempt the sync before timing out
RECREATE_TABLES = False      # True = drop & recreate ALL tables on the endpoint (nuclear option)
POLL_INTERVAL_SECONDS = 2    # How often to poll the LRO status
MAX_POLL_ATTEMPTS = 300      # Safety cap on polling iterations

# Logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("sql_endpoint_sync")

---
## 2. Resolve Workspace ID & Lakehouse ID

When running inside a Fabric notebook, `sempy.fabric` can resolve the current workspace and lakehouse automatically. You can also pass explicit names or GUIDs.

In [ ]:
# --- Option A: Auto-detect from the current notebook context ---
workspace_id = fabric.get_workspace_id()
lakehouse_id = fabric.get_lakehouse_id()

# --- Option B: Specify explicitly (uncomment and fill in) ---
# workspace_id = "xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx"
# lakehouse_id = "xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx"

logger.info(f"Workspace ID : {workspace_id}")
logger.info(f"Lakehouse ID : {lakehouse_id}")

---
## 3. Initialize the Fabric REST Client

`FabricRestClient` handles authentication automatically inside a Fabric notebook session — no manual token management required.

In [ ]:
client = fabric.FabricRestClient()
logger.info("FabricRestClient initialized.")

---
## 4. Get the SQL Analytics Endpoint ID

Every Lakehouse has an associated SQL analytics endpoint. We retrieve its ID from the Lakehouse metadata via:

```
GET /v1/workspaces/{workspaceId}/lakehouses/{lakehouseId}
```

The SQL endpoint ID lives at `response.properties.sqlEndpointProperties.id`.

In [ ]:
def get_sql_endpoint_id(client, workspace_id: str, lakehouse_id: str) -> str:
    """Retrieve the SQL analytics endpoint ID for a given Lakehouse."""
    response = client.get(f"/v1/workspaces/{workspace_id}/lakehouses/{lakehouse_id}")
    response.raise_for_status()
    
    lh_info = response.json()
    sql_props = lh_info.get("properties", {}).get("sqlEndpointProperties", {})
    
    sql_endpoint_id = sql_props.get("id")
    provisioning_status = sql_props.get("provisioningStatus", "Unknown")
    
    if not sql_endpoint_id:
        raise ValueError(
            f"No SQL endpoint found for Lakehouse '{lakehouse_id}'. "
            f"Provisioning status: {provisioning_status}"
        )
    
    logger.info(f"SQL Endpoint ID : {sql_endpoint_id}")
    logger.info(f"Provisioning    : {provisioning_status}")
    logger.info(f"Connection      : {sql_props.get('connectionString', 'N/A')}")
    
    return sql_endpoint_id


sql_endpoint_id = get_sql_endpoint_id(client, workspace_id, lakehouse_id)

---
## 5. Force Metadata Sync

This is the core function. It:

1. **POSTs** a `refreshMetadata` command to the SQL endpoint
2. Handles both **synchronous (200)** and **asynchronous (202 LRO)** responses
3. Polls the operation until it reaches a terminal state (`Succeeded` / `Failed`)
4. Retrieves the per-table sync results

### How the Sync Works Internally
- Fabric scans the Delta log for each table in the Lakehouse
- Tables that are **already in sync** return `NotRun` (no work needed)
- Tables with **new/changed data** are synced and return `Success`
- Tables with **unsupported column types** (ARRAY, MAP, STRUCT) sync partially with warnings
- Tables with **naming conflicts** (e.g., case-only differences) return `Failure` with corrective action

In [ ]:
def refresh_sql_endpoint(
    client,
    workspace_id: str,
    sql_endpoint_id: str,
    recreate_tables: bool = False,
    timeout_unit: str = "Minutes",
    timeout_value: int = 15,
    poll_interval: int = 2,
    max_poll_attempts: int = 300,
) -> dict:
    """
    Force a metadata refresh on the SQL analytics endpoint.

    Parameters
    ----------
    client : FabricRestClient
    workspace_id : str
    sql_endpoint_id : str
    recreate_tables : bool
        If True, drops and recreates ALL tables. Use only for resolving
        persistent inconsistencies.
    timeout_unit : str
        One of: Seconds, Minutes, Hours, Days
    timeout_value : int
        Duration for the timeout.
    poll_interval : int
        Seconds between LRO status checks.
    max_poll_attempts : int
        Safety cap on polling iterations.

    Returns
    -------
    dict with keys: 'status', 'tables' (list of per-table results), 'raw_response'
    """
    url = f"/v1/workspaces/{workspace_id}/sqlEndpoints/{sql_endpoint_id}/refreshMetadata"
    payload = {
        "recreateTables": recreate_tables,
        "timeout": {
            "timeUnit": timeout_unit,
            "value": timeout_value,
        },
    }

    logger.info(f"Triggering metadata refresh (recreateTables={recreate_tables}) ...")
    start_time = time.time()
    response = client.post(url, json=payload)

    # ---- Synchronous completion (200) ----
    if response.status_code == 200:
        logger.info("Sync completed synchronously (HTTP 200).")
        result = response.json()
        return {
            "status": "Succeeded",
            "tables": result.get("value", []),
            "raw_response": result,
            "elapsed_seconds": round(time.time() - start_time, 1),
        }

    # ---- Asynchronous (202 Accepted) — Long Running Operation ----
    if response.status_code == 202:
        operation_url = response.headers.get("Location", "")
        operation_id = response.headers.get("x-ms-operation-id", "")
        retry_after = int(response.headers.get("Retry-After", poll_interval))

        logger.info(f"Async operation started. Operation ID: {operation_id}")
        logger.info(f"Polling every {retry_after}s (max {max_poll_attempts} attempts) ...")

        for attempt in range(1, max_poll_attempts + 1):
            time.sleep(retry_after)

            # Poll operation state
            if operation_url:
                poll_resp = client.get(operation_url)
            else:
                poll_resp = client.get(f"/v1/operations/{operation_id}")

            poll_resp.raise_for_status()
            poll_data = poll_resp.json()

            status = poll_data.get("status", "Unknown")
            pct = poll_data.get("percentComplete", "?")

            if attempt % 5 == 0 or status not in ("Running", "NotStarted"):
                logger.info(f"  Poll #{attempt}: status={status}, progress={pct}%")

            if status == "Succeeded":
                elapsed = round(time.time() - start_time, 1)
                logger.info(f"Sync completed successfully in {elapsed}s.")

                # Fetch the operation result (per-table details)
                result_resp = client.get(f"/v1/operations/{operation_id}/result")
                result_resp.raise_for_status()
                result = result_resp.json()

                return {
                    "status": "Succeeded",
                    "tables": result.get("value", []),
                    "raw_response": result,
                    "elapsed_seconds": elapsed,
                }

            if status == "Failed":
                elapsed = round(time.time() - start_time, 1)
                error_info = poll_data.get("error", {})
                logger.error(f"Sync FAILED after {elapsed}s: {json.dumps(error_info, indent=2)}")
                return {
                    "status": "Failed",
                    "tables": [],
                    "raw_response": poll_data,
                    "elapsed_seconds": elapsed,
                }

        # Exhausted polling attempts
        logger.warning(f"Polling timed out after {max_poll_attempts} attempts.")
        return {
            "status": "PollingTimeout",
            "tables": [],
            "raw_response": poll_data,
            "elapsed_seconds": round(time.time() - start_time, 1),
        }

    # ---- Unexpected status code ----
    response.raise_for_status()  # Will raise for 4xx/5xx
    return {"status": f"Unexpected_{response.status_code}", "tables": [], "raw_response": None}

---
## 6. Execute the Sync

In [ ]:
result = refresh_sql_endpoint(
    client=client,
    workspace_id=workspace_id,
    sql_endpoint_id=sql_endpoint_id,
    recreate_tables=RECREATE_TABLES,
    timeout_unit=TIMEOUT_UNIT,
    timeout_value=TIMEOUT_VALUE,
    poll_interval=POLL_INTERVAL_SECONDS,
    max_poll_attempts=MAX_POLL_ATTEMPTS,
)

logger.info(f"Overall status: {result['status']} ({result.get('elapsed_seconds', '?')}s)")

---
## 7. Per-Table Sync Report

The API returns a status for every table in the Lakehouse:

| Status | Meaning |
|---|---|
| `Success` | Table was synced (new or changed data detected) |
| `NotRun` | Table was already in sync — no work needed |
| `Failure` | Sync failed — check the `error` field for details |

In [ ]:
def format_datetime(iso_str: str) -> str:
    """Convert ISO 8601 datetime to a readable format, or return 'N/A'."""
    if not iso_str:
        return "N/A"
    try:
        dt = datetime.fromisoformat(iso_str.replace("Z", "+00:00"))
        return dt.strftime("%Y-%m-%d %H:%M:%S UTC")
    except (ValueError, TypeError):
        return str(iso_str)


def build_sync_report(tables: list) -> pd.DataFrame:
    """Build a DataFrame summarizing per-table sync results."""
    rows = []
    for t in tables:
        error = t.get("error", {})
        rows.append({
            "Table": t.get("tableName", "?"),
            "Status": t.get("status", "?"),
            "Last Successful Sync": format_datetime(t.get("lastSuccessfulSyncDateTime")),
            "Start": format_datetime(t.get("startDateTime")),
            "End": format_datetime(t.get("endDateTime")),
            "Error Code": error.get("errorCode", ""),
            "Error Message": error.get("message", ""),
        })

    df = pd.DataFrame(rows)
    if not df.empty:
        # Sort: Failures first, then Success, then NotRun
        status_order = {"Failure": 0, "Success": 1, "NotRun": 2}
        df["_sort"] = df["Status"].map(status_order).fillna(3)
        df = df.sort_values("_sort").drop(columns="_sort").reset_index(drop=True)
    return df


if result["tables"]:
    report_df = build_sync_report(result["tables"])

    # Summary counts
    counts = report_df["Status"].value_counts().to_dict()
    print(f"\n{'='*60}")
    print(f"  SYNC SUMMARY")
    print(f"  Total tables : {len(report_df)}")
    print(f"  Synced       : {counts.get('Success', 0)}")
    print(f"  Already OK   : {counts.get('NotRun', 0)}")
    print(f"  Failed       : {counts.get('Failure', 0)}")
    print(f"{'='*60}\n")

    display(report_df)
else:
    print("No table-level results returned.")

---
## 8. Inspect Failures & Warnings (Detail View)

If any tables failed to sync, this cell shows the full error details from the raw API response.

In [ ]:
failures = [t for t in result["tables"] if t.get("status") == "Failure"]

if failures:
    print(f"\n{'!'*60}")
    print(f"  {len(failures)} TABLE(S) FAILED TO SYNC")
    print(f"{'!'*60}\n")
    for t in failures:
        print(f"  Table: {t.get('tableName')}")
        error = t.get("error", {})
        print(f"    Error Code     : {error.get('errorCode', 'N/A')}")
        print(f"    Message        : {error.get('message', 'N/A')}")
        print(f"    Related Resource: {error.get('relatedResource', 'N/A')}")
        print()
else:
    print("No failures detected.")

# Show full raw JSON for deeper inspection
print("\n--- Raw API Response (for debugging) ---")
print(json.dumps(result["raw_response"], indent=2, default=str))

---
## 9. (Optional) Pipeline Integration Helper

Use this function to call the sync after a Spark write operation in a data pipeline. It wraps the sync with retry logic and raises on failure so pipeline orchestration can catch it.

In [ ]:
def sync_after_write(
    wait_before_sync: int = 5,
    max_retries: int = 2,
    recreate_on_retry: bool = False,
):
    """
    Call this after writing Delta tables to force SQL endpoint sync.

    Parameters
    ----------
    wait_before_sync : int
        Seconds to wait after the Spark write before triggering sync.
        Helps avoid race conditions with Delta log commits.
    max_retries : int
        Number of retry attempts on failure.
    recreate_on_retry : bool
        If True, sets recreateTables=True on retry attempts.
    """
    _client = fabric.FabricRestClient()
    _ws_id = fabric.get_workspace_id()
    _lh_id = fabric.get_lakehouse_id()
    _sql_id = get_sql_endpoint_id(_client, _ws_id, _lh_id)

    logger.info(f"Waiting {wait_before_sync}s before sync ...")
    time.sleep(wait_before_sync)

    for attempt in range(1, max_retries + 1):
        recreate = recreate_on_retry and attempt > 1
        r = refresh_sql_endpoint(
            client=_client,
            workspace_id=_ws_id,
            sql_endpoint_id=_sql_id,
            recreate_tables=recreate,
        )

        if r["status"] == "Succeeded":
            failed_tables = [t["tableName"] for t in r["tables"] if t.get("status") == "Failure"]
            if not failed_tables:
                logger.info("All tables synced successfully.")
                return r
            logger.warning(f"Tables with failures: {failed_tables}")

        if attempt < max_retries:
            logger.info(f"Retrying ({attempt}/{max_retries}) ...")
            time.sleep(5)

    raise RuntimeError(
        f"SQL endpoint sync failed after {max_retries} attempts. "
        f"Last status: {r['status']}. Check the raw_response for details."
    )


# Example usage (uncomment to run after a Spark write):
# spark.sql("CREATE OR REPLACE TABLE my_table AS SELECT 1 AS id, 'test' AS name")
# sync_result = sync_after_write(wait_before_sync=5)

---
## Common Issues & Troubleshooting

| Issue | Cause | Fix |
|---|---|---|
| Table not appearing in SQL endpoint | Sync delay | Run this notebook to force sync |
| `Failure` with case-sensitivity error | Two tables differ only by case (e.g., `Test` vs `test`) | Rename one table — SQL endpoint is case-insensitive |
| Columns missing after sync | `ARRAY`, `MAP`, or `STRUCT` columns | These types are unsupported in SQL endpoint; use scalar columns |
| `OperationHasNoResult` (HTTP 400) | LRO completed but result expired | Increase `poll_interval`, or re-run |
| Persistent stale data | Corrupted metadata cache | Set `RECREATE_TABLES = True` and re-run (drops & recreates all tables) |

### Notes
- The sync is **incremental** — tables already in sync (`NotRun`) are skipped
- `recreateTables=True` is the nuclear option — it drops and rebuilds everything on the SQL endpoint
- This API requires **Contributor** or higher workspace role
- The API is officially documented as of 2025 (it was previously undocumented/community-sourced)